In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_csv("diabetes.csv")
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [5]:
X=df.iloc[:,:-1]
y=df.iloc[:,-1]


In [6]:
scaler=StandardScaler()
X=scaler.fit_transform(X)
X.shape

(768, 8)

In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test,y_train,y_test= train_test_split(X,y,test_size=0.2, random_state=42)

In [103]:
import tensorflow
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout

In [17]:
model=Sequential()
model.add(Dense(32,activation="relu",input_dim=8))
model.add(Dense(1,activation="sigmoid"))

model.compile(optimizer="adam",loss="binary_crossentropy", metrics=["accuracy"])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
model.fit(X_train,y_train, batch_size=32,epochs=10,validation_data=(X_test,y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8111 - loss: 0.3941 - val_accuracy: 0.7597 - val_loss: 0.5417
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8127 - loss: 0.3936 - val_accuracy: 0.7662 - val_loss: 0.5406
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8176 - loss: 0.3934 - val_accuracy: 0.7662 - val_loss: 0.5414
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.3930 - val_accuracy: 0.7597 - val_loss: 0.5412
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8094 - loss: 0.3928 - val_accuracy: 0.7597 - val_loss: 0.5414
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.3922 - val_accuracy: 0.7468 - val_loss: 0.5422
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8127 - loss: 0.3921 - val_accuracy: 0.7662 - val_loss: 0.5406
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8111 - loss: 0.3913 - val_accuracy: 0.7662 - val_loss

In [43]:
import kerastuner as kt


In [44]:
def build_model(hp):
  model=Sequential()
  model.add(Dense(32,activation="relu",input_dim=8))
  model.add(Dense(1,activation="sigmoid"))
  optimizer=hp.Choice("Optimizer",values=["adam","sgd","rmsprop","adadelta"])
  model.compile(optimizer=optimizer, loss='binary_crossentropy',metrics=["accuracy"])
  return model

In [48]:
tuner=kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Reloading Tuner from ./untitled_project/tuner0.json


In [49]:
tuner.get_best_hyperparameters()[0].values

{'Optimizer': 'adam'}

In [56]:
model=tuner.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [57]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)

In [59]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.7166 - loss: 0.5844 - val_accuracy: 0.7143 - val_loss: 0.5684
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7248 - loss: 0.5543 - val_accuracy: 0.7403 - val_loss: 0.5488
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7394 - loss: 0.5318 - val_accuracy: 0.7468 - val_loss: 0.5354
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7492 - loss: 0.5157 - val_accuracy: 0.7338 - val_loss: 0.5265
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7590 - loss: 0.5031 - val_accuracy: 0.7532 - val_loss: 0.5198
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7687 - loss: 0.4930 - val_accuracy: 0.7468 - val_loss: 0.5178
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7655 - loss: 0.4860 - val_accuracy: 0.7597 - val_loss: 0.5164
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7704 - loss: 0.4791 - val_accuracy: 0.75

In [75]:
def build_model1(hp):
  model=Sequential()
  units=hp.Int('units',min_value=8,max_value=128,step=8)
  model.add(Dense(units=units,activation="relu", input_dim=8))
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer='adam', loss='binary_crossentropy',metrics=['accuracy'])
  return model

In [76]:
tuner2=kt.RandomSearch(build_model1, objective="val_accuracy",max_trials=5,directory='mydir')

In [80]:
tuner2.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7597402334213257

Best val_accuracy So Far: 0.7922077775001526
Total elapsed time: 00h 00m 14s


In [81]:
tuner2.get_best_hyperparameters()[0].values

{'units': 104}

In [82]:
model=tuner2.get_best_models(num_models=1)[0]

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [83]:
model.fit(X_train,y_train,batch_size=32,epochs=100,initial_epoch=6)

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7622 - loss: 0.4686
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7687 - loss: 0.4588 
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7704 - loss: 0.4522 
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7720 - loss: 0.4483 
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7785 - loss: 0.4455 
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7850 - loss: 0.4410 
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7899 - loss: 0.4387 
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7964 - loss: 0.4358
Epoch 15/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7997 - loss: 0.4335 
Epoch 16/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7964 - loss: 0.4324 
Epoch 17/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8013 - loss: 0.4298 
Epoch 18/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/s

In [84]:
def build_model(hp):
  model=Sequential()
  model.add(Dense(104,activation='relu',input_dim=8))
  for i in range(hp.Int('num_layers',min_value=1, max_value=10)):
    model.add(Dense(104, activation="relu"))
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"])
  return model

In [85]:
tuner=kt.RandomSearch(build_model,objective="val_accuracy",max_trials=5,directory='mydir',project_name="num_layers")
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.7727272510528564

Best val_accuracy So Far: 0.8116883039474487
Total elapsed time: 00h 00m 23s


In [86]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 4}

In [88]:
model=tuner.get_best_models(num_models=1)[0]
model.fit(X_train,y_train,epochs=100,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.7671 - loss: 0.4895 - val_accuracy: 0.7273 - val_loss: 0.5276
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7818 - loss: 0.4492 - val_accuracy: 0.7662 - val_loss: 0.5152
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7883 - loss: 0.4323 - val_accuracy: 0.7662 - val_loss: 0.5241
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8094 - loss: 0.4135 - val_accuracy: 0.7597 - val_loss: 0.5460
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8029 - loss: 0.4016 - val_accuracy: 0.7208 - val_loss: 0.5688
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8274 - loss: 0.3956 - val_accuracy: 0.7662 - val_loss: 0.5427
Epoch 13/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8225 - loss: 0.3953 - val_accuracy: 0.7403 - val_loss: 0.5760
Epoch 14/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8143 - loss: 0.3854 - val_accuracy: 0.72

In [109]:
def build_model(hp):
  model=Sequential()
  counter=0
  for i in range(hp.Int('num_layers',min_value=1, max_value=10)):
    if counter==0:
      model.add(Dense(hp.Int('unit'+str(i),min_value=8,max_value=128,step=8),
                    activation=hp.Choice('activatin'+str(i),values=['relu','tanh','sigmoid']),
                    input_dim=8))
      model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    else:
      model.add(Dense(hp.Int('unit'+str(i),min_value=8,max_value=128,step=8),
                    activation=hp.Choice('activatin'+str(i),values=['relu','tanh','sigmoid']),
                    input_dim=8))
      model.add(Dropout(hp.Choice('dropout'+str(i),values=[0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9])))
    counter+=1
  model.add(Dense(1,activation="sigmoid"))
  model.compile(optimizer=hp.Choice("optimizer",values=['rmsprop','adam','sgd','nadam','adadelta']),
                  loss="binary_crossentropy",
                  metrics=['accuracy'])
  return model



In [110]:
tuner=kt.RandomSearch(build_model,objective="val_accuracy",max_trials=5,directory='mydir',project_name='final_parameters2')

In [111]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 5 Complete [00h 00m 03s]
val_accuracy: 0.6428571343421936

Best val_accuracy So Far: 0.6428571343421936
Total elapsed time: 00h 00m 20s


In [112]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 8,
 'unit0': 24,
 'activatin0': 'tanh',
 'dropout0': 0.6,
 'optimizer': 'rmsprop',
 'unit1': 8,
 'activatin1': 'relu',
 'dropout1': 0.1,
 'unit2': 8,
 'activatin2': 'relu',
 'dropout2': 0.1,
 'unit3': 8,
 'activatin3': 'relu',
 'dropout3': 0.1,
 'unit4': 8,
 'activatin4': 'relu',
 'dropout4': 0.1,
 'unit5': 8,
 'activatin5': 'relu',
 'dropout5': 0.1,
 'unit6': 8,
 'activatin6': 'relu',
 'dropout6': 0.1,
 'unit7': 8,
 'activatin7': 'relu',
 'dropout7': 0.1}

In [113]:
model=tuner.get_best_models(num_models=1)[0]
model.fit(X_train,y_train,epochs=200,initial_epoch=6,validation_data=(X_test,y_test))

Epoch 7/200


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.6531 - loss: 0.6535 - val_accuracy: 0.6429 - val_loss: 0.6458
Epoch 8/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.6329 - val_accuracy: 0.6429 - val_loss: 0.6293
Epoch 9/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.6233 - val_accuracy: 0.6429 - val_loss: 0.6111
Epoch 10/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.6137 - val_accuracy: 0.6429 - val_loss: 0.6049
Epoch 11/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6515 - loss: 0.5959 - val_accuracy: 0.6429 - val_loss: 0.5887
Epoch 12/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.5900 - val_accuracy: 0.6429 - val_loss: 0.5763
Epoch 13/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.5970 - val_accuracy: 0.6429 - val_loss: 0.5772
Epoch 14/200
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6531 - loss: 0.5944 - val_accuracy: 0.6429 - val_los